In [6]:
import sys 
import sklearn
import tensorflow as tf
from tensorflow import keras

import numpy as np
import os 
np.random.seed(42)
tf.random.set_seed(42)
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

In [9]:
import requests
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response=requests.get(url)
with open('tiny_shakespeare.txt', 'w', encoding='utf-8') as f:
    f.write(response.text)
with open('tiny_shakespeare.txt') as f:
    shakespeare_text = f.read()

In [10]:
print(shakespeare_text[:148])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?



In [11]:
tokenizer=keras.preprocessing.text.Tokenizer(char_level=True)
tokenizer.fit_on_texts(shakespeare_text)

In [12]:
tokenizer.texts_to_sequences(["First"])

[[20, 6, 9, 8, 3]]

In [13]:
tokenizer.sequences_to_texts([[20, 6, 9, 8, 3]])

['f i r s t']

In [14]:
max_id=len(tokenizer.word_index)
dataset_size=tokenizer.document_count

In [15]:
[encoded]=np.array(tokenizer.texts_to_sequences([shakespeare_text]))-1
train_size=dataset_size*90//100
dataset=tf.data.Dataset.from_tensor_slices(encoded[:train_size])

In [16]:
n_steps=100
window_length=n_steps+1
dataset=dataset.window(window_length,shift=1,drop_remainder=True)

In [17]:
dataset=dataset.flat_map(lambda window:window.batch(window_length))

In [18]:
np.random.seed(42)
tf.random.set_seed(42)


In [19]:
batch_size=32
dataset=dataset.shuffle(10000).batch(batch_size)
dataset=dataset.map(lambda windows:(windows[:,:-1],windows[:,1:]))

In [20]:
dataset=dataset.map(
    lambda X_batch,Y_batch:(tf.one_hot(X_batch,depth=max_id),Y_batch)
)

In [21]:
dataset=dataset.prefetch(1)

In [22]:
for X_batch,Y_batch in dataset.take(1):
    print(X_batch.shape,Y_batch.shape)

(32, 100, 39) (32, 100)


In [ ]:
model = keras.models.Sequential([
    keras.layers.GRU(128, return_sequences=True, input_shape=[None, max_id],
                     #dropout=0.2, recurrent_dropout=0.2),
                     dropout=0.2),
    keras.layers.GRU(128, return_sequences=True,
                     #dropout=0.2, recurrent_dropout=0.2),
                     dropout=0.2),
    keras.layers.TimeDistributed(keras.layers.Dense(max_id,
                                                    activation="softmax"))
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
history = model.fit(dataset, epochs=10)